In [ ]:
import tensorflow.compat.v1 as tf

tf.config.set_soft_device_placement(True)
# tf.debugging.set_log_device_placement(True)
# from sklearn.metrics import confusion_matrix
import numpy as np
# from scipy.io import loadmat
import os
# from functools import reduce
# from scipy import signal
import pandas as pd
# from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow import keras as K
from tensorflow.keras.layers import Add, Dense, Activation, Flatten, concatenate, Input, Dropout, LSTM, Conv2D, MaxPooling2D, Conv1D, MaxPooling1D, BatchNormalization, PReLU, ELU, ReLU
from tensorflow.keras.models import Sequential, Model, load_model
import matplotlib.pyplot as plt
# import datetime
# get_ipython().run_line_magic('load_ext', 'tensorboard')
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# get_ipython().run_line_magic('matplotlib', 'inline')
# from collections import Counter 
np_load_old = np.load
# from tensorflow.keras.utils import plot_model
from tensorflow.keras.initializers import glorot_uniform
# modify the default parameters of np.load
np.load = lambda *a,**k: np_load_old(*a, allow_pickle=True, **k)
np.random.seed(42)
tf.set_random_seed(42)
from sklearn.metrics import classification_report
import argparse
import h5py

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import itertools

In [ ]:
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)
    plt.figure(figsize=(15,10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=36)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, fontsize=30)
    plt.yticks(tick_marks, classes, fontsize=30)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black", fontsize=28)

    plt.tight_layout()
    plt.ylabel('True label', fontsize=30)
    plt.xlabel('Predicted label', fontsize=30)

## Load in data

In [ ]:
# load diffusion-augmented train set + original validation/test sets
loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'
os.makedirs(os.path.join(loadPath, "models"), exist_ok=True)

bestModel = '/home/sz4544/EEG-motor-imagery-main/project/models/cnn_tr1800_diffusion_aug_2CNN.keras'

f_train = h5py.File(os.path.join(loadPath, "train1800_diffusion_aug.h5"), "r")
tr_data = f_train['data'][:]
ytr = f_train['tasks'][:]

f_valid = h5py.File(os.path.join(loadPath, "valid360_raw_EEG.h5"), "r")
val_data = f_valid['data'][:]
yval = f_valid['tasks'][:]
val_subjects = f_valid['subjects'][:]

f_test = h5py.File(os.path.join(loadPath, "test360_raw_EEG.h5"), "r")
ts_data = f_test['data'][:]
yts = f_test['tasks'][:]
ts_subjects = f_test['subjects'][:]

print("train:", tr_data.shape, ytr.shape)
print("valid:", val_data.shape, yval.shape)
print("test:", ts_data.shape, yts.shape)
print("train labels:", np.unique(ytr))
print("valid labels:", np.unique(yval))
print("test labels:", np.unique(yts))

In [ ]:
print("train label counts:", np.unique(ytr, return_counts=True))
print("valid label counts:", np.unique(yval, return_counts=True))
print("test label counts:", np.unique(yts, return_counts=True))

## Normalization

In [ ]:
# flatten and reshape data
xtr_s_flattened = np.squeeze(tr_data).ravel().reshape((-1, 64))
xval_s_flattened = np.squeeze(val_data).ravel().reshape((-1, 64))
xts_s_flattened = np.squeeze(ts_data).ravel().reshape((-1, 64))
print(xtr_s_flattened.shape)
print(xval_s_flattened.shape)
print(xts_s_flattened.shape)

# normalize data
scaler = StandardScaler()
Ztr_temp = scaler.fit_transform(xtr_s_flattened)
Zval_temp = scaler.transform(xval_s_flattened)
Zts_temp = scaler.transform(xts_s_flattened)

# flatten and reshape data back
Ztr = np.squeeze(Ztr_temp).ravel().reshape((-1, 640, 64))
Zval = np.squeeze(Zval_temp).ravel().reshape((-1, 640, 64))
Zts = np.squeeze(Zts_temp).ravel().reshape((-1, 640, 64))
print(Ztr.shape)
print(Zval.shape)
print(Zts.shape)

In [ ]:
print("normalized real-like samples mean/std:")
for i in [0, 1, 100, 500]:
    print(i, np.mean(Ztr[i]), np.std(Ztr[i]), ytr[i])

print("normalized aug-like samples mean/std:")
for i in [1800, 1801, 1850, 2000, 2500]:
    print(i, np.mean(Ztr[i]), np.std(Ztr[i]), ytr[i])

In [ ]:
# unify labels to 0,1,2,3
ytr = ytr.astype(np.int64)
yval = yval.astype(np.int64)
yts = yts.astype(np.int64)

if np.array_equal(np.unique(ytr), np.array([1, 2, 3, 4])):
    ytr = ytr - 1

if np.array_equal(np.unique(yval), np.array([1, 2, 3, 4])):
    yval = yval - 1

if np.array_equal(np.unique(yts), np.array([1, 2, 3, 4])):
    yts = yts - 1

print("after remap")
print("train labels:", np.unique(ytr))
print("valid labels:", np.unique(yval))
print("test labels:", np.unique(yts))

In [ ]:
x_train = Ztr[..., np.newaxis].astype(np.float32)
x_valid = Zval[..., np.newaxis].astype(np.float32)
x_test = Zts[..., np.newaxis].astype(np.float32)

y_train = pd.get_dummies(ytr).values.astype(np.float32)
y_valid = pd.get_dummies(yval).values.astype(np.float32)
y_test = pd.get_dummies(yts).values.astype(np.float32)

print("x_train:", x_train.shape, x_train.dtype)
print("x_valid:", x_valid.shape, x_valid.dtype)
print("x_test:", x_test.shape, x_test.dtype)

print("y_train:", y_train.shape, y_train.dtype)
print("y_valid:", y_valid.shape, y_valid.dtype)
print("y_test:", y_test.shape, y_test.dtype)

## Model architecture

In [ ]:
strategy = tf.distribute.MirroredStrategy()
print('Number of devices: {}'.format(strategy.num_replicas_in_sync))

In [ ]:
with strategy.scope():
    nodes = 640
    inputs = Input(shape=(nodes, 64, 1))

    conv1 = Conv2D(8, (15,9), strides=(2,1), kernel_initializer = glorot_uniform(seed=42))(inputs)
    batch1 = BatchNormalization()(conv1)
    prelu1 = PReLU()(batch1)
    maxpool1 = MaxPooling2D((2,2))(prelu1)
    conv2 = Conv2D(16, (15,9), strides=(2,1), kernel_initializer = glorot_uniform(seed=42))(maxpool1)
    batch2 = BatchNormalization()(conv2)
    prelu2 = PReLU()(batch2)
    # maxpool2 = MaxPooling2D((5,1))(prelu2)

    flat = Flatten()(prelu2)
    dense1 = Dense(16, activation='sigmoid', kernel_initializer = glorot_uniform(seed=42))(flat)
    drop1 = Dropout(0.2)(dense1)
    predictions = Dense(4, activation='softmax', kernel_initializer = glorot_uniform(seed=42))(drop1)
    model = Model(inputs=inputs,outputs=predictions)
print(model.summary())

In [ ]:
with strategy.scope():
    opt_adam = K.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999, epsilon=1e-08)
    model.compile(loss='categorical_crossentropy', optimizer=opt_adam, metrics=['categorical_accuracy'])

es = EarlyStopping(monitor='val_categorical_accuracy', mode='max', verbose=1, patience=30)
mc = ModelCheckpoint(bestModel, monitor='val_categorical_accuracy', mode='max', verbose=1, save_best_only=True)

In [ ]:
history = model.fit(x=x_train, y=y_train, epochs=300, batch_size=32,shuffle=True,
                    verbose=1, validation_data = (x_valid, y_valid), callbacks=[es, mc])

## Model testing

In [ ]:
best_model = tf.keras.models.load_model(bestModel)
yhat = np.argmax(best_model.predict(x_test), axis=1)
ytrue = np.argmax(y_test, axis=1)

print("bestModel:", bestModel)
print('test acc: {:.2f}%'.format(np.sum(yhat == ytrue) / yts.shape[0] * 100))

In [ ]:
target_names = ['open one hand', 'imagine open one hand', 'open both hands/feet', 'imagine open both hands/feet']
print(classification_report(ytrue, yhat, target_names=target_names, digits=4))

In [ ]:
cm = confusion_matrix(ytrue, yhat)
cm

In [ ]:
plot_confusion_matrix(cm, ['Task1', 'Task2', 'Task3', 'Task4'], normalize=True, title='Confusion matrix', cmap=plt.cm.Greys)

In [ ]:
print("train data shape:", tr_data.shape)

print("first 5 real-like samples mean/std:")
for i in [0, 1, 2, 100, 500]:
    print(i, np.mean(tr_data[i]), np.std(tr_data[i]), ytr[i])

print("first 5 aug-like samples mean/std:")
for i in [1800, 1801, 1850, 2000, 2500]:
    print(i, np.mean(tr_data[i]), np.std(tr_data[i]), ytr[i])

In [ ]:
import matplotlib.pyplot as plt

real_idx = 0
aug_idx = 1800

plt.figure(figsize=(14,4))
plt.plot(tr_data[real_idx, :, 0], label=f"real idx={real_idx}, y={ytr[real_idx]}")
plt.plot(tr_data[aug_idx, :, 0], label=f"aug idx={aug_idx}, y={ytr[aug_idx]}")
plt.legend()
plt.title("Channel 0: real vs augmented")
plt.show()

In [ ]:
for aug_idx in [1800, 1850, 2000, 2500]:
    plt.figure(figsize=(14,4))
    plt.plot(tr_data[0, :, 0], label=f"real idx=0, y={ytr[0]}")
    plt.plot(tr_data[aug_idx, :, 0], label=f"aug idx={aug_idx}, y={ytr[aug_idx]}")
    plt.legend()
    plt.title(f"Channel 0 compare: aug idx={aug_idx}")
    plt.show()